# Serving a Machine Learning Model with FastAPI
## Part 2 · Building a client

The server from [Part 1](server.ipynb) is only useful if something talks to it. In this notebook you write that something in Python with the [`requests`](https://requests.readthedocs.io/) library, going from a single raw HTTP call to a small reusable client with proper error handling.

> **Before you start:** the Part 1 server must be running (`start_server()` in `server.ipynb`) and listening on `http://127.0.0.1:8000`.

```text
  this notebook                         Part 1 server
 +----------------+  1. GET  /models     +----------------------+
 | DetectionClient| -------------------> |                      |
 |  (requests)    |  2. POST /predict/*  |  FastAPI + YOLO      |
 |                | -------------------> |                      |
 |                | <------------------- |                      |
 +----------------+  JSON  /  JPEG bytes +----------------------+
```

**What you will learn**

- How a request URL is built: host, port, path, query string
- Sending **GET** and **POST** requests with `requests`, including **file uploads** (`multipart/form-data`)
- Reading status codes, headers, JSON and binary bodies
- Wrapping raw HTTP in a reusable client: sessions, timeouts, error handling
- Using the same API from the terminal with `curl`

**Requirements:** Python 3.9+, the running Part 1 server, and a few JPG/PNG images (the same `images/` folder as Part 1; sample photos are fetched automatically if it is empty).

| Section | Topic |
|---|---|
| 1 | Setup |
| 2 | Anatomy of a request |
| 3 | Raw calls: GET, then POST with a file |
| 4 | A reusable client class |
| 5 | Putting the API to work |
| 6 | When things go wrong: error handling |
| 7 | The same calls with `curl` |
| 8 | Cheat sheet and takeaways |

## 1 · Setup

In [ ]:
%pip install -q requests

In [ ]:
import statistics
import time
import urllib.request
from collections import Counter
from pathlib import Path
from urllib.parse import parse_qs, urlsplit

import requests
from IPython.display import Image, display

In [ ]:
BASE_URL = "http://127.0.0.1:8000"     # where the Part 1 server listens
IMAGES_DIR = Path("images")            # shared with Part 1

SAMPLE_URLS = [                        # public sample photos (Ultralytics YOLOv5 repo, AGPL-3.0), fetched at run time
    "https://raw.githubusercontent.com/ultralytics/yolov5/master/data/images/bus.jpg",
    "https://raw.githubusercontent.com/ultralytics/yolov5/master/data/images/zidane.jpg",
]
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}


def list_images(folder: Path = IMAGES_DIR) -> list[Path]:
    return sorted(p for p in folder.glob("*") if p.suffix.lower() in IMAGE_SUFFIXES)


IMAGES_DIR.mkdir(exist_ok=True)
if not list_images():
    for url in SAMPLE_URLS:
        urllib.request.urlretrieve(url, IMAGES_DIR / url.rsplit("/", 1)[-1])

image_paths = list_images()
print("Images:", [p.name for p in image_paths])

## 2 · Anatomy of a request

Everything a client does is built from a URL plus a few extras. Python's standard library can take a URL apart, which makes the pieces easy to see:

In [ ]:
url = f"{BASE_URL}/predict/json?model=yolov4-tiny&confidence=0.5"
parts = urlsplit(url)

print("scheme :", parts.scheme)                # protocol
print("host   :", parts.hostname)              # which machine
print("port   :", parts.port)                  # which program on that machine
print("path   :", parts.path)                  # which endpoint
print("query  :", parse_qs(parts.query))       # parameters after '?'

| Piece | Example | Role |
|---|---|---|
| scheme | `http` | the protocol (`https` = encrypted) |
| host | `127.0.0.1` | the machine; `127.0.0.1` / `localhost` means "this computer" |
| port | `8000` | the program listening on that machine |
| path | `/predict/json` | the endpoint, i.e. which function runs |
| query | `?model=...&confidence=...` | small named parameters |

Query parameters carry small, simple values. A file does not fit in a URL, so it travels in the request **body** as `multipart/form-data`. With `requests` you never build either by hand: `params=` URL-encodes the query string and `files=` builds the multipart body.

## 3 · Raw calls

### GET: read something

A `GET` needs no payload. Start with the endpoints that report on the server itself, and inspect every part of the response:

> A `ConnectionError` here means the server is not running: start it in Part 1.

In [ ]:
response = requests.get(f"{BASE_URL}/models", timeout=5)

print("status :", response.status_code, response.reason)
print("headers:", dict(response.headers))
print("body   :", response.json())

Always pass a **`timeout`**. Without one, `requests` waits forever if the server stops answering and your program hangs.

### POST: send an image, receive JSON

Here `params=` fills the query string and `files=` uploads the image. The key `"file"` must match the parameter name the server declared (`file`), otherwise the server answers 422.

In [ ]:
image_path = image_paths[0]

with open(image_path, "rb") as f:                         # "rb": images are binary data
    response = requests.post(
        f"{BASE_URL}/predict/json",
        params={"model": "yolov4-tiny", "confidence": 0.5},   # -> ?model=yolov4-tiny&confidence=0.5
        files={"file": f},                                    # -> multipart/form-data body
        timeout=30,
    )

sent = response.request                                   # the request exactly as it went over the wire
print("sent     :", sent.method, sent.url)
print("sent     :", sent.headers["Content-Type"])           # multipart/form-data; boundary=...
print("received :", response.status_code, response.headers["Content-Type"])
print()
response.json()

The JSON holds the model used, the number of objects and, per object, a `label`, a `confidence` and a `box` (`[x1, y1, x2, y2]` in pixels).

### POST: send an image, receive an image

The other endpoint returns the picture with boxes drawn on it. The body is now **binary**, not JSON, so read `response.content`. Metadata can still travel in headers (here, `X-Object-Count`).

In [ ]:
with open(image_path, "rb") as f:
    response = requests.post(f"{BASE_URL}/predict/image", params={"confidence": 0.5}, files={"file": f}, timeout=30)

print("status :", response.status_code)
print("type   :", response.headers["Content-Type"])
print("size   :", f"{len(response.content) / 1024:.0f} KB")
print("objects:", response.headers["X-Object-Count"])
display(Image(data=response.content, format="jpeg", width=480))

**Reading a response**

| Attribute | Gives you | Use it for |
|---|---|---|
| `.status_code`, `.ok` | the number / `True` for any 2xx or 3xx | did it work? |
| `.headers` | case-insensitive dict | content type, custom metadata |
| `.json()` | parsed Python object | JSON bodies |
| `.text` | decoded string | HTML, plain text, error pages |
| `.content` | raw `bytes` | images, files |
| `.raise_for_status()` | raises on 4xx / 5xx | quick failure handling |

## 4 · A reusable client class

The raw calls repeat the same boilerplate. A small class removes it and gives every call the same behaviour:

- a **`Session`** reuses the connection between calls (faster than reconnecting each time),
- a default **timeout**,
- **one place** that turns HTTP errors into a Python exception carrying the server's explanation,
- parameters are only sent when given, so the **server owns the defaults** and the client never duplicates them.

In [ ]:
class APIError(Exception):
    """The server answered with an error status (4xx or 5xx)."""

    def __init__(self, status_code: int, detail):
        super().__init__(f"HTTP {status_code}: {detail}")
        self.status_code = status_code
        self.detail = detail


class DetectionClient:
    """Thin Python wrapper around the object-detection API."""

    def __init__(self, base_url: str = BASE_URL, timeout: float = 30):
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout                    # seconds
        self.session = requests.Session()

    def _request(self, method: str, path: str, **kwargs) -> requests.Response:
        """Send a request; raise APIError for any error status."""
        response = self.session.request(method, self.base_url + path, timeout=self.timeout, **kwargs)
        if not response.ok:
            try:
                detail = response.json()["detail"]    # FastAPI puts its explanation under "detail"
            except (ValueError, KeyError):
                detail = response.text
            raise APIError(response.status_code, detail)
        return response

    def _predict(self, endpoint: str, image_path, model, confidence) -> requests.Response:
        wanted = {"model": model, "confidence": confidence}
        params = {k: v for k, v in wanted.items() if v is not None}      # omit what was not specified
        with open(image_path, "rb") as f:
            return self._request("POST", endpoint, params=params, files={"file": f})

    # --- public API -----------------------------------------------------------
    def health(self) -> dict:
        return self._request("GET", "/").json()

    def models(self) -> dict:
        return self._request("GET", "/models").json()

    def detect(self, image_path, model=None, confidence=None) -> dict:
        """Detections as a dict: {'model', 'count', 'detections': [...]}."""
        return self._predict("/predict/json", image_path, model, confidence).json()

    def annotate(self, image_path, model=None, confidence=None) -> bytes:
        """The image with boxes drawn on it, as JPEG bytes."""
        return self._predict("/predict/image", image_path, model, confidence).content

In [ ]:
client = DetectionClient()
print(client.health())
print(client.models())

result = client.detect(image_paths[0])
print(f"\n{result['count']} objects found by {result['model']}")

## 5 · Putting the API to work

### Use the JSON in your own logic

Returning structured data is what makes the API useful to programs: count objects, filter them, trigger actions.

In [ ]:
result = client.detect(image_paths[0], confidence=0.3)

print("objects per class :", dict(Counter(d["label"] for d in result["detections"])))

# Filtering happens client-side: ask once with a low threshold, then try stricter ones locally
# without another round-trip. (This is why the response includes the scores.)
for threshold in (0.3, 0.6, 0.9):
    kept = [d for d in result["detections"] if d["confidence"] >= threshold]
    print(f"confidence >= {threshold}: {len(kept)} objects")

### Process a whole folder

Annotated images come back as bytes: write them to disk, or display them directly.

In [ ]:
OUTPUT_DIR = Path("images_predicted")
OUTPUT_DIR.mkdir(exist_ok=True)

for path in image_paths:
    jpeg = client.annotate(path)
    out_path = OUTPUT_DIR / f"{path.stem}_detected.jpg"
    out_path.write_bytes(jpeg)                          # bytes -> file
    print("saved", out_path)
    display(Image(data=jpeg, format="jpeg", width=360))

### See the effect of the threshold

The `confidence` parameter from Part 1, now driven through the API:

In [ ]:
THRESHOLDS = (0.8, 0.5, 0.2)

print(f"{'image':<20}" + "".join(f"conf>{t:<7}" for t in THRESHOLDS))
for path in image_paths:
    counts = [client.detect(path, confidence=t)["count"] for t in THRESHOLDS]
    print(f"{path.name[:18]:<20}" + "".join(f"{c:<12}" for c in counts))

### Measure latency

A client sees the **round-trip** time: upload + inference + response. Always measure several calls; the first is often slower.

In [ ]:
timings = []
for _ in range(10):
    start = time.perf_counter()
    client.detect(image_paths[0])
    timings.append((time.perf_counter() - start) * 1000)

print(f"10 calls: median {statistics.median(timings):.0f} ms, slowest {max(timings):.0f} ms")

### Optional: compare the two models

Set the flag to `True` to run both models on the same image. The first request to the full model makes the **server download ~250 MB**, so the timeout is raised and the call is repeated once to exclude that start-up cost from the timing.

In [ ]:
COMPARE_MODELS = False

if COMPARE_MODELS:
    client.timeout = 900                                   # the first call may include a large download
    for name in client.models()["models"]:
        client.detect(image_paths[0], model=name)          # warm-up: downloads / loads the model
        start = time.perf_counter()
        result = client.detect(image_paths[0], model=name)
        elapsed = (time.perf_counter() - start) * 1000
        print(f"{name:<12} {result['count']} objects, {elapsed:.0f} ms round-trip")
    client.timeout = 30

## 6 · When things go wrong

A robust client distinguishes three kinds of failure:

| Kind | What you see | Whose fault | What to do |
|---|---|---|---|
| **No answer** | a `requests` exception (`ConnectionError`, `Timeout`) | network or server down | retry with a delay, then report |
| **4xx status** | `APIError` with a `detail` | the **request** is wrong | fix it; **retrying will not help** |
| **5xx status** | `APIError` | the **server** failed | retry later, report if it persists |

Provoke each one deliberately. The helper below reports what happened instead of crashing the notebook:

In [ ]:
def attempt(label: str, call) -> None:
    """Run `call` and print its outcome."""
    try:
        call()
        print(f"{label:<26} -> succeeded")
    except APIError as e:
        detail = e.detail
        if isinstance(detail, list):        # validation errors (422): a list of {loc, msg, ...}
            detail = "; ".join(f"{'.'.join(map(str, d['loc']))}: {d['msg']}" for d in detail)
        print(f"{label:<26} -> HTTP {e.status_code}: {detail}")
    except requests.exceptions.RequestException as e:      # no HTTP response at all
        print(f"{label:<26} -> {type(e).__name__} (no response from server)")


not_an_image = Path("not_an_image.txt")
not_an_image.write_text("definitely not an image")

attempt("valid request", lambda: client.detect(image_paths[0]))
attempt("text file sent as image", lambda: client.detect(not_an_image))                    # 415
attempt("unknown model name", lambda: client.detect(image_paths[0], model="yolov99"))      # 422
attempt("confidence out of range", lambda: client.detect(image_paths[0], confidence=2))    # 422
attempt("server not reachable", lambda: DetectionClient("http://127.0.0.1:59999", timeout=3).health())

not_an_image.unlink()                                      # clean up

Note how much the server's validation did for free: the 415 and both 422 responses come with a precise, machine-readable explanation, and the Python code on the server never ran for the invalid requests.

For simple scripts, `response.raise_for_status()` is the shortest way to turn a 4xx/5xx into an exception. The class above goes one step further and keeps the server's `detail` message.

## 7 · The same calls with `curl`

Any language, or the terminal, can use the API: it is just HTTP. `-F` sends a `multipart/form-data` upload, `-o` saves the response body to a file.

```bash
# GET: list models
curl http://127.0.0.1:8000/models

# POST: image in, JSON out
curl -X POST "http://127.0.0.1:8000/predict/json?model=yolov4-tiny&confidence=0.5" \
     -F "file=@images/bus.jpg"

# POST: image in, annotated image out
curl -X POST "http://127.0.0.1:8000/predict/image" \
     -F "file=@images/bus.jpg" -o result.jpg
```

The interactive docs at `http://127.0.0.1:8000/docs` and the specification at `/openapi.json` describe the same contract; tools can generate client libraries in other languages from the latter.

## 8 · Cheat sheet and takeaways

| Task | Code |
|---|---|
| GET and parse JSON | `requests.get(url, timeout=5).json()` |
| POST a file with parameters | `requests.post(url, params={...}, files={"file": f}, timeout=30)` |
| Check the outcome | `r.ok`, `r.status_code`, `r.raise_for_status()` |
| Read the body | `r.json()` · `r.text` · `r.content` (bytes) |
| Read headers | `r.headers["Content-Type"]` |
| Reuse connections | `s = requests.Session(); s.post(...)` |

**Key takeaways**

1. An API call is a **method + URL + parameters + optional body**; the response is a **status + headers + body**.
2. Let `requests` build the request (`params=`, `files=`); never assemble query strings or multipart bodies by hand.
3. **Always set a timeout** and **always check the status** before using the body.
4. Wrap raw HTTP in a small **client class**: one place for configuration, errors and defaults.
5. Handle the three failure kinds differently: retry network errors and 5xx, fix the request on 4xx.

**Going further**

- **Concurrency:** send many images in parallel with `httpx.AsyncClient` or a thread pool.
- **Resilience:** automatic retries with back-off (`urllib3.util.Retry` mounted on a `Session`).
- **Security:** authentication headers (`headers={"Authorization": "Bearer ..."}`) once the server requires them.
- **Real data:** feed video frames or camera snapshots to the same endpoints.

When you have finished, stop the server with `stop_server()` in [Part 1](server.ipynb).